<!-- nav -->
[← 11 · A parts catalog: schemas that build on schemas](11_A_Parts_Catalog.ipynb) · [Home](../../README.md) · [Equivalence of the implementations →](../../docs/EQUIVALENCE.md)

# 12 · Keeping history: provenance, journals and snapshots

**The problem.** A team discusses its work on a board of comments. Someone asks who wrote a comment, when, and from
where, and who changed it last. A second service must hold the same comments, written by the same transactions. And a
restart must bring everything back as it was, history included, without adding history of its own.

Three things answer these:

1. **Provenance**, an extension, records each transaction that writes: who, when, in which session and where, and,
   for the schemas the app chooses, which objects each transaction created and which it modified last
   ([CAP-134: transactions recorded with provenance](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-134.md)).
2. A **journal** is a list of transactions' expressions, written as data; another store replays it by executing them,
   and makes the same objects and the same records
   ([CAP-129: journals replayed by execution](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-129.md)).
3. A **snapshot** writes a whole store, its schemas and its data, as two transactions, which a store reads back
   ([CAP-127: snapshots made of transactions](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-127.md)).

In [1]:
import { Builders, JSON as Json, Plain, Provenance, References } from "@mbse/schemas/Framework";

/** A team's board, a singleton owning its comments, each given its ID by the board. */
function install(txn: any): any {
  const S = txn.Schemas; // the store's own schemas, by name
  return txn.uid("tutorial.board/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Board: S.OfObject, Posted: S.OfRelation, Comment: S.OfObject })
    .spec([
      (_: any) => _.Posted.name("work.Posted").links([(t: any) => t.name("board"), (t: any) => t.name("comment")])
        .properties([(t: any) => t.name("id").schema([S.OfStr]).pattern(String.raw`C-(\d+)`)])
        .uniques([["board", "id"], ["comment"]]),
      (_: any) => _.Board.name("work.Board").ref(true).singleton("work.TheBoard").adjacencies({
        comments: (a: any) => a.relation([_.Posted]).me("board") }),
      (_: any) => _.Comment.name("work.Comment").ref(true).properties({ text: S.OfStr }).adjacencies([
        (t: any) => t.name("board").relation([_.Posted]).me("comment").owner(true)]),
    ])]))]);
}

The board owns each comment through `work.Posted`, keyed by its `id`, which has a naming pattern (case study 6): a
comment built anywhere without one is placed on the board, the one singleton that can own it, under the next ID,
`C-1`, `C-2`. An owned object has a path, its owner and its key, so a transaction in another store, or on another day,
can refer to it.

<a id="T12.1"></a>
## 12.1 · Who changed what: provenance

`Provenance` is installed after the app's own install, and `Provenance.extend` names the schemas whose objects it
links to their records, here comments. A transaction then says who writes, when and from where: `when` is UTC ISO 8601
text, the session has an id and the identity it belongs to (`who`), a locale and a device, and `where` a latitude and
longitude. A service sets them from the request it serves; the record is made from them, so that a replay makes the
same one. A transaction built first, with `store.Mutations.OfTransaction`, is run as any is. `post` names the comment
it writes by its identity, never by its text: the board, a singleton, and the comment's `id` there, which the request
gives, so that it finds the same comment whenever it runs (12.3):

In [2]:
/** A store of the board, recording the history of its comments. */
function made(): any {
  const store: any = new Builders.Store();
  store.transaction(install);
  store.transaction(Provenance.install); // after the app's own install
  store.transaction((t: any) => Provenance.extend(t, [t.work.Comment])); // the schemas whose history it keeps
  return store;
}

const store = made();
const board = store.singleton("work.TheBoard");

/** A transaction posting a comment, as a service writes it: who, when and where, from the request; the comment named
 * by its identity, the board and its `id`. */
function post(uid: string, when: string, id: string, text: string): any {
  return store.Mutations.OfTransaction((t: any) => t.uid(uid).when(when)
    .session((s: any) => s.id("tab-7f3a").who("ada").locale("en-GB"))
    .where((w: any) => w.latitude(51.5).longitude(-0.125))
    .mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ b: store.work.Board, c: store.work.Comment })
      .spec([(_: any) => _.b.comments({ [id]: (e: any) => e.comment(_.c) }), (_: any) => _.c.text(text)])]))]));
}

const first = post("post-1", "2026-10-26T14:30:05Z", "C-1", "Should chips show adjacencies?");
const comment = store.transaction(first)[0].get("c");
console.log([...board.comments].map((e: any) => [`${e.id}`, `${e.comment.text}`]));

[ [ "C-1", "Should chips show adjacencies?" ] ]


The records are data in the same store, each in a directory singleton keyed by name: transactions by their `uid`,
sessions by their id, identities by their name. The record holds what the transaction said:

In [3]:
/** The keys of a directory singleton's entries. */
function keys(name: string, adjacency: string, key: string): string[] {
  return [...store.singleton(name)[adjacency]].map((e: any) => `${e[key]}`).sort();
}

const record = [...store.singleton("Provenance.Transactions").transactions].find((e: any) => `${e.uid}` === "post-1").transaction;
const where = store.member(record, "where");
console.log(keys("Provenance.Transactions", "transactions", "uid"), `${record.when}`,
  [store.member(where, "latitude"), store.member(where, "longitude")]);
console.log(keys("Provenance.Sessions", "sessions", "id"), keys("Provenance.Identities", "identities", "name"));

[ "post-1" ] 2026-10-26T14:30:05Z [ 51.5, -0.125 ]


[ "tab-7f3a" ] [ "ada" ]


<a id="T12.2"></a>
## 12.2 · An object's records, read from the record's side

Ada edits her comment in a second transaction, which refers to it by its reference (case study 10). The links from a
comment to its records, `CreatedBy` and `ModifiedBy`, are anonymous adjacencies: a comment has no member for them, so
its schema, and every program that reads it, is unchanged
([CAP-178: an object's provenance read through its own named adjacencies](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-178.md) plans named ones). Read
them from the record's side, with a map (case study 10) whose symbol is given the comment's reference: the record that
`created` it, its session and the session's identity; and the record that `modified` it, the latest write only.

In [4]:
/** A transaction writing a comment's text, the comment given by its reference. */
function edit(uid: string, when: string, text: string): any {
  return store.Mutations.OfTransaction((t: any) => t.uid(uid).when(when)
    .session((s: any) => s.id("tab-7f3a").who("ada"))
    .mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ c: References.of(store, comment) })
      .spec([(_: any) => _.c.text(text)])]))]));
}

const second = edit("edit-2", "2026-10-26T15:00:00Z", "Should chips show their adjacencies?");
store.transaction(second);
const created = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
  .symbols({ c: References.of(store, comment), t: store.Provenance.OfTransaction,
    s: store.Provenance.OfSession, i: store.Provenance.OfIdentity })
  .spec([(_: any) => _.t.created([_.c]), (_: any) => _.t.session([_.s]), (_: any) => _.s.who([_.i])])]))]));
const modified = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
  .symbols({ c: References.of(store, comment), t: store.Provenance.OfTransaction })
  .spec([(_: any) => _.t.modified([_.c])])]))]));
const frame = created[0]; // one frame: the record, its session and the identity
console.log(created.length, `${frame.get("t").when}`, `${frame.get("s").locale}`, [...frame.get("i").directory].map((e: any) => `${e.name}`));
console.log(modified.map((f: any) => [...f.get("t").directory].map((e: any) => `${e.uid}`)));

1 2026-10-26T14:30:05Z en-GB [ "ada" ]


[ [ "edit-2" ] ]


A write made imperatively, within `store.transaction(() => ...)`, has no expression to say who made it. Its record
holds placeholders: a random UUID for its uid, the clock's time, and the session and identity `unknown`. A service
that knows its caller writes an expression instead.

In [5]:
store.transaction(() => {
  store.work.Comment(comment, (c: any) => c.text("Should chips show their adjacencies, Bob?"));
});
const uids = keys("Provenance.Transactions", "transactions", "uid");
const uuid = /^[0-9a-f]{8}-[0-9a-f]{4}-4[0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}$/;
console.log(uids.length, uids.filter((u) => !uuid.test(u)), keys("Provenance.Sessions", "sessions", "id"));

3 [ "edit-2", "post-1" ] [ "tab-7f3a", "unknown" ]


<a id="T12.3"></a>
## 12.3 · A journal, replayed in another store

A transaction's expression is data: `Plain.ToPlain(store).Transaction(tx)` writes it, and
`Plain.FromPlain(other).Transaction(data)` reads it into a store, to run there. A **journal** is a list of them, kept
in order. The second service makes a store of the same installs and replays the journal: each expression finds what it
names or makes it, and its record is made from what it says, so the records are the same. Replayed whole again, in
order, the journal writes nothing twice, because each write names the comment by its key, never by its text: `post` by
the board and `C-1`, `edit` by its reference, the same path. The second `post` finds the comment whose text `edit`
changed, and writes its text again rather than make another; each record is found by its uid.

In [6]:
const journal = [first, second].map((tx) => Json.dumps(Plain.ToPlain(store).Transaction(tx))); // as text, to send or keep
console.log(journal[1]!.slice(0, 220), "...");

const again = made();
for (let replay = 0; replay < 2; replay++) { // the whole journal, in order, twice: the second time, nothing written twice
  for (const entry of journal) {
    again.transaction(Plain.FromPlain(again).Transaction(Json.loads(entry)));
  }
}
console.log([...again.singleton("work.TheBoard").comments].map((e: any) => [`${e.id}`, `${e.comment.text}`]));
console.log([...again.singleton("Provenance.Transactions").transactions].map((e: any) => `${e.uid}`).sort());

{"uid": "edit-2", "when": "2026-10-26T15:00:00Z", "session": {"id": "tab-7f3a", "who": "ada"}, "mutations": [{"upsert": {"exists": [{"symbols": {"c": {"reference": {"path": {"singleton": "work.TheBoard", "next": {"entry" ...


[ [ "C-1", "Should chips show their adjacencies?" ] ]


[ "edit-2", "post-1" ]


An upsert finds an object by its identity, a singleton, or an owned object's owner and key, never by a property that
may change: a `post` that found its comment by its text would find nothing once `edit` had changed it, and the journal
replayed again would make a second comment
([CAP-139: upserts find before they create](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-139.md)).

<a id="T12.4"></a>
## 12.4 · A snapshot, and a restart

`Plain.ToPlain(store).Snapshot()`, and its JSON and YAML forms, write a whole store as two transactions: its schemas,
each in its module, and then the data its singletons reach, the provenance records included. Reading it back executes
them: a schema the reading store holds is found by its name, and the rest are made. A restart writes a snapshot and
reads it into a new store; the records come back, and reading adds none, since what a snapshot writes is the history it
holds, not a change of its own. Written again, the snapshot is the same.

In [7]:
const text = Json.ToJSON(store).Snapshot();
const restarted = made();
Json.FromJSON(restarted).Snapshot(text);
console.log([...restarted.singleton("work.TheBoard").comments].map((e: any) => `${e.comment.text}`));
console.log(Json.dumps([...restarted.singleton("Provenance.Transactions").transactions].map((e: any) => `${e.uid}`).sort()) === Json.dumps(uids));
console.log(Json.ToJSON(restarted).Snapshot() === text);

[ "Should chips show their adjacencies, Bob?" ]


true


true


## What you learned

- `Provenance`, installed after the app's own install, records each transaction that writes; `Provenance.extend`
  links the objects of the schemas it names to the records that created them and that modified them last.
- A transaction's expression gives the record: `uid`, `when`, `session` (`id`, `who`, `locale`, `device`) and
  `where`. An imperative write's record holds placeholders.
- An object's records are read from the record's side, with a map whose symbol is given the object's reference.
- A journal is a list of expressions, written and read with `Plain`'s `Transaction`; replayed in order in another
  store, it makes the same objects and records, and replayed again, nothing twice, since each write names its object
  by its identity (a singleton, or an owner and key), never by a property that may change.
- A snapshot writes a whole store, schemas and data, as two transactions; read back, it restores the records and adds
  none.

**Next:** the design in mbse-basis ([Schema-driven data](https://github.com/pitaman71/mbse-basis/blob/main/docs/features/README.md#schema-driven-data)) is the reference for everything here, and
[mbse-expressions](https://github.com/pitaman71/mbse-expressions) continues where case study 10 stops.

<!-- nav -->
[← 11 · A parts catalog: schemas that build on schemas](11_A_Parts_Catalog.ipynb) · [Home](../../README.md) · [Equivalence of the implementations →](../../docs/EQUIVALENCE.md)